# Lab 05-08 · Measuring growth rates by counting operations

Companion to chapter [05-08](../../chapters/05-08-data-structures-and-algorithms.md). Handout: [README.md](README.md).

Timings vary from machine to machine and run to run; **operation counts** do not. This notebook counts comparisons and probes as the input size $n$ grows, and reads growth rates off log-log plots. Cells marked **🔮 Predict** ask for a prediction first. All random data use fixed seeds.

In [1]:
import math
import random
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

## Part 1 — Searching

**🔮 Predict:** on a log-log plot of comparisons against $n$, what slope does linear search have? And binary search?

In [2]:
def linear_comparisons(a, target):
    for k, x in enumerate(a, 1):
        if x == target:
            return k
    return len(a)

def binary_comparisons(a, target):
    lo, hi, c = 0, len(a) - 1, 0
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        c += 1
        if a[mid] == target:
            return c
        if a[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return c

rng = random.Random(1)
sizes = [2**k for k in range(4, 17, 2)]
lin, bina = [], []
for n in sizes:
    a = list(range(n))
    targets = [rng.randrange(n) for _ in range(200)]
    lin.append(np.mean([linear_comparisons(a, t) for t in targets]))
    bina.append(np.mean([binary_comparisons(a, t) for t in targets]))
slope_lin = np.polyfit(np.log(sizes), np.log(lin), 1)[0]
print(f"linear search: slope {slope_lin:.2f}; at n = {sizes[-1]}: {lin[-1]:.0f} comparisons on average")
print(f"binary search at n = {sizes[-1]}: {bina[-1]:.1f} comparisons on average (max {math.floor(math.log2(sizes[-1])) + 1})")

linear search: slope 0.99; at n = 65536: 32289 comparisons on average
binary search at n = 65536: 15.2 comparisons on average (max 17)


Linear search has slope 1 on log-log axes ($\text{cost} \propto n^1$), averaging about $n/2$. Binary search grows so slowly that a log-log slope is nearly 0; plotted against $\log_2 n$ it is a straight line. Its average, about $\log_2 n - 1$ (15.2 at $n = 65{,}536$), sits about two comparisons below the maximum $\lfloor\log_2 n\rfloor + 1$, because half of all targets are found only at the deepest level.

## Part 2 — Sorting

**🔮 Predict:** the slope of insertion sort's comparison count, and of merge sort's ($n\log n$ is "slightly more than 1").

In [3]:
def insertion_comparisons(a):
    a, c = list(a), 0
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0:
            c += 1
            if a[j] <= x:
                break
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = x
    return c

def merge_comparisons(a):
    if len(a) <= 1:
        return list(a), 0
    left, cl = merge_comparisons(a[:len(a) // 2])
    right, cr = merge_comparisons(a[len(a) // 2:])
    out, i, j, c = [], 0, 0, cl + cr
    while i < len(left) and j < len(right):
        c += 1
        if left[i] <= right[j]:
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    return out + left[i:] + right[j:], c

rng = random.Random(2)
ns = [64, 128, 256, 512, 1024, 2048]
ins, mer = [], []
for n in ns:
    data = [rng.random() for _ in range(n)]
    ins.append(insertion_comparisons(data))
    mer.append(merge_comparisons(data)[1])
s_ins = np.polyfit(np.log(ns), np.log(ins), 1)[0]
s_mer = np.polyfit(np.log(ns), np.log(mer), 1)[0]
print(f"insertion sort slope {s_ins:.2f}; merge sort slope {s_mer:.2f}")
print(f"n = 2048: insertion {ins[-1]}, n(n-1)/4 = {2048 * 2047 // 4}; merge {mer[-1]}, log2(n!) = {math.log2(math.factorial(2048)):.0f}")
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(ns, ins, "o-", label="insertion sort")
ax.loglog(ns, mer, "s-", label="merge sort")
ax.loglog(ns, [n * math.log2(n) for n in ns], "--", label=r"$n\log_2 n$")
ax.set_xlabel("n"); ax.set_ylabel("comparisons"); ax.legend(); fig.tight_layout()
plt.show()

insertion sort slope 2.00; merge sort slope 1.21
n = 2048: insertion 1081333, n(n-1)/4 = 1048064; merge 19954, log2(n!) = 19580


Insertion sort's slope is 2.00; merge sort's is a little above 1: the local slope of $n\log n$ is $1 + 1/\ln n$, about 1.17 over this range, and merge sort's lower-order $-n$ term steepens it a little more. At $n = 2048$, insertion sort needs about 54 times more comparisons than merge sort, and the gap widens in proportion to $n/\log n$. Merge sort stays within a few percent of the information-theoretic minimum $\log_2 n!$.

## Part 3 — Hash-table probes against load factor

**🔮 Predict:** at load factor 0.9, how many probes does an unsuccessful search take with linear probing? (Use the chapter's formula.)

In [4]:
def probe_stats(m, alpha, rng):
    table = [None] * m
    keys = rng.sample(range(10**9), int(alpha * m))
    succ = 0
    for k in keys:
        i, p = hash((k, 7)) % m, 1                 # a well-mixing hash of the key
        while table[i] is not None:
            i, p = (i + 1) % m, p + 1
        table[i] = k
        succ += p                                  # finding k later retraces these probes
    unsucc = 0
    for _ in range(2000):
        i, p = rng.randrange(m), 1                 # an absent key's home slot is uniformly random
        while table[i] is not None:
            i, p = (i + 1) % m, p + 1
        unsucc += p
    return succ / len(keys), unsucc / 2000

rng = random.Random(3)
print(" alpha  successful (theory)   unsuccessful (theory)")
for alpha in (0.5, 0.75, 0.9):
    s, u = probe_stats(2**16, alpha, rng)
    print(f" {alpha:.2f}   {s:5.2f} ({0.5 * (1 + 1 / (1 - alpha)):5.2f})      {u:6.2f} ({0.5 * (1 + 1 / (1 - alpha) ** 2):6.2f})")

 alpha  successful (theory)   unsuccessful (theory)
 0.50    1.50 ( 1.50)        2.50 (  2.50)
 0.75    2.45 ( 2.50)        8.20 (  8.50)
 0.90    5.14 ( 5.50)       46.09 ( 50.50)


The measurements track Knuth's formulas: modest at 0.5, noticeably worse at 0.75, and dramatically worse for misses at 0.9, where clusters of occupied slots grow long. (The unsuccessful counts are noisier because a few long clusters dominate them.) This is why the lab's C map resizes before the load passes 0.75.

## Your turn

1. In Part 2, sort data that are already sorted, and data sorted in reverse. How do the two algorithms' counts change, and why?
2. In Part 3, replace linear probing with "double hashing" (a second hash sets the step size) and compare the unsuccessful-search counts at 0.9.